## Condition Configuration

This notebook prepares reusable data artifacts for the selected condition.

- `CONDITION`: condition name under `data/ai-proposals/` and `data/ai-proposals/rephrased/`.
- `BASE_CONDITION`: raw proposal/review condition name before adding the `rephrased/` prefix.


In [1]:
CONDITION = 'minimal'
BASE_CONDITION = 'minimal'


# Prepare Data for Analysis

This notebook creates reusable prepared artifacts for the rephrased/minimal analyses. It deliberately stops before analysis-specific metrics such as diversity, novelty scores, outlier flags, and metric_score_df.

Outputs:
- prepared proposal records: original + rephrased text, no analysis-derived metrics
- NCEMS all-reviews table and novelty all-reviews table
- full rephrased proposal embeddings
- abstract-only rephrased proposal embeddings for PubMed/literature comparisons
- literature abstract embeddings
- review embeddings for NCEMS and novelty reviews


## 1. Setup

In [2]:
import json
import os
import pickle
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel


def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    for candidate in candidates:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate.resolve()
    raise RuntimeError('Could not find project root containing src/ and data/.')


PROJECT_ROOT = find_project_root()
condition = f'rephrased/{CONDITION}'
base_condition = BASE_CONDITION

RESULTS_DIR = PROJECT_ROOT / 'results'
TABLES_DIR = RESULTS_DIR / 'tables' / condition
PREPARED_DIR = TABLES_DIR / 'prepared'
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

AI_ORIGINAL_DIR = PROJECT_ROOT / 'data' / 'ai-proposals' / base_condition
AI_REPHRASED_DIR = PROJECT_ROOT / 'data' / 'ai-proposals' / condition
HUMAN_ORIGINAL_DIR = PROJECT_ROOT / 'data' / 'human-proposals'
HUMAN_REPHRASED_DIR = PROJECT_ROOT / 'data' / 'human-proposals' / 'rephrased'

NCEMS_AI_REVIEWS_DIR = PROJECT_ROOT / 'data' / 'reviews' / 'ai_reviews' / base_condition / 'ncems_criteria' / 'rephrased'
NOVELTY_AI_REVIEWS_DIR = PROJECT_ROOT / 'data' / 'reviews' / 'ai_reviews' / base_condition / 'novelty'
HUMAN_REVIEWS_DIR = PROJECT_ROOT / 'data' / 'reviews' / 'human_reviews' / 'rephrased'
LITERATURE_PATH = PROJECT_ROOT / 'data' / 'literature' / 'relevant-corpus-from-pubmed.json'

ALL_PROPOSALS_PATH = TABLES_DIR / 'all_proposals.json'
PREPARED_ALL_PROPOSALS_PATH = PREPARED_DIR / 'all_proposals.json'
PREPARED_ALL_PROPOSALS_CSV = PREPARED_DIR / 'all_proposals.csv'
NCEMS_ALL_REVIEWS_PATH = PREPARED_DIR / 'ncems_criteria_all_reviews.csv'
NOVELTY_ALL_REVIEWS_PATH = PREPARED_DIR / 'novelty_all_reviews.csv'
REVIEW_SCORES_WIDE_PATH = TABLES_DIR / 'review_scores_wide.csv'

EMBEDDINGS_DIR = PROJECT_ROOT / 'data' / 'embeddings'
PROPOSAL_FULL_EMBEDDINGS_FILE = EMBEDDINGS_DIR / condition / 'proposal_embeddings_human_ai_rephrased.pkl'
PROPOSAL_ABSTRACT_EMBEDDINGS_FILE = EMBEDDINGS_DIR / condition / 'proposal_embeddings_section1_only.pkl'
LITERATURE_EMBEDDINGS_FILE = EMBEDDINGS_DIR / 'literature' / 'relevant_literature_embeddings.pkl'
NCEMS_REVIEW_EMBEDDINGS_FILE = EMBEDDINGS_DIR / 'reviews' / base_condition / 'ncems_criteria' / f'review_embeddings_{base_condition}.pkl'
NOVELTY_REVIEW_EMBEDDINGS_FILE = EMBEDDINGS_DIR / 'reviews' / base_condition / 'novelty' / f'review_embeddings_{base_condition}.pkl'

for p in [PROPOSAL_FULL_EMBEDDINGS_FILE, PROPOSAL_ABSTRACT_EMBEDDINGS_FILE, LITERATURE_EMBEDDINGS_FILE, NCEMS_REVIEW_EMBEDDINGS_FILE, NOVELTY_REVIEW_EMBEDDINGS_FILE]:
    p.parent.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Prepared tables: {PREPARED_DIR}')

Project root: /Users/eveyhuang/Documents/NICO/human-AI-proposal
Prepared tables: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/prepared


## 2. Shared Helpers

In [3]:
SECTION_HEADERS = [
    'SCIENTIFIC BACKGROUND AND RESEARCH QUESTION',
    'METHODOLOGY AND ANALYTICAL APPROACH',
    'DATA SOURCES AND SYNTHESIS PLAN',
    'FEASIBILITY AND TIMELINE',
    'OPEN SCIENCE AND TEAM COMPOSITION',
]


def normalize_title(title):
    if pd.isna(title):
        return ''
    title = str(title).strip().lower()
    title = re.sub(r'\s+', ' ', title)
    title = re.sub(r'[^a-z0-9 ]', '', title)
    return title


def strip_section_headers(text):
    text = '' if pd.isna(text) else str(text)
    for header in SECTION_HEADERS:
        text = re.sub(rf'^\s*{re.escape(header)}\s*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    return re.sub(r'\n{3,}', '\n\n', text).strip()


def first_existing(paths, description):
    paths = sorted(paths)
    if not paths:
        raise FileNotFoundError(f'No {description} found')
    return paths[-1]


def safe_text(value):
    return '' if pd.isna(value) else str(value)


def proposal_abstract_text(row):
    candidates = [
        row.get('abstract'),
        row.get('section1'),
        row.get('scientific_background_and_research_question'),
        row.get('main_idea'),
    ]
    for value in candidates:
        text = safe_text(value).strip()
        if text:
            return strip_section_headers(text)
    full = strip_section_headers(row.get('standardized_text', ''))
    return full.split('\n\n', 1)[0].strip() if full else ''

## 3. Build Prepared Proposal Records

In [4]:
ai_original_path = first_existing(AI_ORIGINAL_DIR.glob('ai_proposals_minimal_complete_*.csv'), f'original AI proposal CSV in {AI_ORIGINAL_DIR}')
ai_rephrased_path = first_existing(AI_REPHRASED_DIR.glob('ai_proposals_minimal_rephrased_*.csv'), f'rephrased AI proposal CSV in {AI_REPHRASED_DIR}')

ai_original = pd.read_csv(ai_original_path)
ai_rephrased = pd.read_csv(ai_rephrased_path)
ai_rephrased['_key'] = ai_rephrased['title'].map(normalize_title)
ai_rephrased_lut = {r['_key']: r for _, r in ai_rephrased.iterrows()}

AI_ORIGINAL_SECTIONS = [
    'abstract', 'background_and_significance', 'research_questions_and_hypotheses',
    'methods_and_approach', 'expected_outcomes_and_impact', 'budget_and_resources',
]

records = []
for _, row in ai_original.iterrows():
    key = normalize_title(row.get('title', ''))
    rep = ai_rephrased_lut.get(key, {})
    standardized_text = safe_text(rep.get('standardized_text', ''))
    abstract_text = proposal_abstract_text(rep)
    records.append({
        'title': row.get('title'),
        'title_norm': key,
        'group': row.get('model', 'AI'),
        'is_ai': True,
        'model': row.get('model'),
        'cohort': None,
        'source_file': ai_rephrased_path.name,
        'original': {s: row.get(s) for s in AI_ORIGINAL_SECTIONS},
        'rephrased': {
            'standardized_text': standardized_text,
            'full_text': strip_section_headers(standardized_text),
            'abstract_text': abstract_text,
            'main_idea': rep.get('main_idea'),
        },
        'metrics': {},
    })

for cohort in ['y1', 'y2']:
    original_path = HUMAN_ORIGINAL_DIR / f'human-proposals-{cohort}.json'
    rephrased_path = first_existing(HUMAN_REPHRASED_DIR.glob(f'human_proposals_rephrased_{cohort}_*.json'), f'rephrased human {cohort} proposal JSON in {HUMAN_REPHRASED_DIR}')

    with open(original_path) as f:
        original_payload = json.load(f)
    with open(rephrased_path) as f:
        rephrased_payload = json.load(f)

    rep_lut = {}
    for prop in rephrased_payload.get('proposals', rephrased_payload if isinstance(rephrased_payload, list) else []):
        title = prop.get('proposal_title', prop.get('title', ''))
        rep_lut[normalize_title(title)] = prop

    for prop in original_payload.get('proposals', original_payload if isinstance(original_payload, list) else []):
        title = prop.get('proposal_title', prop.get('title', ''))
        key = normalize_title(title)
        rep = rep_lut.get(key, {})
        standardized_text = safe_text(rep.get('standardized_text', ''))
        abstract_text = proposal_abstract_text(rep)
        records.append({
            'title': title,
            'title_norm': key,
            'group': 'Human',
            'is_ai': False,
            'model': None,
            'cohort': cohort,
            'source_file': rephrased_path.name,
            'original': {
                'abstract': prop.get('abstract'),
                'full_draft': prop.get('full_draft'),
            },
            'rephrased': {
                'standardized_text': standardized_text,
                'full_text': strip_section_headers(standardized_text),
                'abstract_text': abstract_text,
                'main_idea': rep.get('main_idea'),
            },
            'metrics': {},
        })

proposal_df = pd.DataFrame([{
    'title': r['title'],
    'title_norm': r['title_norm'],
    'group': r['group'],
    'is_ai': r['is_ai'],
    'model': r['model'],
    'cohort': r['cohort'],
    'source_file': r['source_file'],
    'standardized_text': r['rephrased']['standardized_text'],
    'full_text': r['rephrased']['full_text'],
    'abstract_text': r['rephrased']['abstract_text'],
    'main_idea': r['rephrased']['main_idea'],
} for r in records])

if proposal_df['title_norm'].duplicated().any():
    dupes = proposal_df.loc[proposal_df['title_norm'].duplicated(keep=False), ['title', 'group', 'cohort']]
    print('Warning: duplicated normalized titles:')
    display(dupes)

with open(PREPARED_ALL_PROPOSALS_PATH, 'w') as f:
    json.dump(records, f, indent=2, default=str)
with open(ALL_PROPOSALS_PATH, 'w') as f:
    json.dump(records, f, indent=2, default=str)
proposal_df.to_csv(PREPARED_ALL_PROPOSALS_CSV, index=False)

print(f'Saved prepared proposals: {PREPARED_ALL_PROPOSALS_PATH}')
print(f'Saved analysis-compatible proposals: {ALL_PROPOSALS_PATH}')
print(f'Saved flat proposal table: {PREPARED_ALL_PROPOSALS_CSV}')
print(proposal_df.groupby(['group', 'is_ai']).size())

Saved prepared proposals: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/prepared/all_proposals.json
Saved analysis-compatible proposals: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/all_proposals.json
Saved flat proposal table: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/prepared/all_proposals.csv
group                 is_ai
Human                 False    23
claude-opus-4-5       True     23
gemini-3-pro-preview  True     23
gpt-5.2               True     23
dtype: int64


## 4. Build Prepared Review Tables

In [5]:
NCEMS_CRITERIA_ORDER = [
    'Relevance_to_Emergent_Phenomena',
    'Novelty_and_Significance',
    'Rigor_of_Approach',
    'Scope_and_Timeline',
    'Synthesis_Focus',
    'Data_Identification',
    'Open_Science_Commitment',
]

NCEMS_CRITERION_MAP = {
    'Relevance to Emergent Phenomena': 'Relevance_to_Emergent_Phenomena',
    'Novelty & Significance': 'Novelty_and_Significance',
    'Rigor of Approach': 'Rigor_of_Approach',
    'Scope & Timeline': 'Scope_and_Timeline',
    'Synthesis Focus': 'Synthesis_Focus',
    'Data Identification': 'Data_Identification',
    'Open Science Commitment': 'Open_Science_Commitment',
}

HUMAN_NCEMS_COL_MAP = {
    'scientific_merit_and_innovation_score': ['Relevance_to_Emergent_Phenomena', 'Novelty_and_Significance', 'Rigor_of_Approach'],
    'feasibility_score': ['Scope_and_Timeline'],
    'data_sources_and_limitations_score': ['Synthesis_Focus', 'Data_Identification'],
    'open_science_compliance_score': ['Open_Science_Commitment'],
}

NOVELTY_CRITERIA_ORDER = [
    'new_question_topic_or_framing',
    'new_theory_concept_method_dataset_or_design',
    'unusual_combination_of_existing_ideas',
    'beyond_state_of_the_art',
    'credible_high_risk_high_gain',
    'unique_knowledge_generation',
]


def build_ncems_ai_reviews():
    path = first_existing(NCEMS_AI_REVIEWS_DIR.glob('ncems_reviews_rephrased*.json'), f'rephrased NCEMS AI reviews in {NCEMS_AI_REVIEWS_DIR}')
    with open(path) as f:
        payload = json.load(f)
    rows = []
    for i, r in enumerate(payload.get('reviews', [])):
        ev = (r.get('evaluations') or {}).get('evaluation')
        if not isinstance(ev, dict):
            continue
        criteria = {k: np.nan for k in NCEMS_CRITERIA_ORDER}
        justifications = []
        for category in ev.get('criteria_scores', []) or []:
            for sc in category.get('subcriteria', []) or []:
                c_name = NCEMS_CRITERION_MAP.get(sc.get('criterion'))
                if c_name is not None:
                    criteria[c_name] = sc.get('score', np.nan)
                j = sc.get('justification')
                if isinstance(j, str) and j.strip():
                    justifications.append(j.strip())
        overall = ev.get('overall_rating') or {}
        overall_score = overall.get('final_numeric_score', np.nan)
        overall_summary = overall.get('narrative_summary', '')
        review_text = safe_text(r.get('rephrased_review') or r.get('rephrased_reviews')).strip()
        if not review_text:
            review_text = '\n\n'.join(justifications + ([overall_summary] if overall_summary else []))
        proposal_id = r.get('proposal_id')
        title = r.get('title')
        rows.append({
            'review_uid': f'ncems-ai-{i:04d}',
            'review_type': 'ncems_criteria',
            'review_source': 'ai',
            'source_file': path.name,
            'proposal_id': proposal_id,
            'title': title,
            'title_norm': normalize_title(title),
            'author': r.get('author'),
            'evaluator': r.get('evaluator'),
            'overall_score': overall_score,
            'review_text': review_text,
            **criteria,
        })
    df = pd.DataFrame(rows)
    if len(df):
        missing = df['proposal_id'].isna() | df['proposal_id'].astype(str).str.strip().str.lower().isin(['', 'n/a', 'na', 'none', 'nan'])
        df.loc[missing, 'proposal_id'] = df.loc[missing, 'title_norm']
        df['proposal_uid'] = df['author'].astype(str) + '::' + df['proposal_id'].astype(str)
    return df


def build_ncems_human_reviews(cohort):
    path = first_existing(HUMAN_REVIEWS_DIR.glob(f'human_reviews_human-{cohort}_rephrased*.csv'), f'rephrased human {cohort} reviews in {HUMAN_REVIEWS_DIR}')
    raw = pd.read_csv(path)
    df = raw.copy()
    if 'rephrased_review' not in df.columns:
        if 'rephrased_reviews' in df.columns:
            df['rephrased_review'] = df['rephrased_reviews']
        else:
            raise KeyError(f"Expected rephrased_review column in {path}")
    df['author'] = f'human-{cohort}'
    df['evaluator'] = df['reviewer_id'].astype(str).map(lambda x: f'human-reviewer-{x}')
    df['title_norm'] = df['title'].map(normalize_title)
    for col in HUMAN_NCEMS_COL_MAP:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['overall_score'] = pd.to_numeric(df['overall_rating_score'], errors='coerce')
    df['overall_score'] = df['overall_score'].fillna(df[list(HUMAN_NCEMS_COL_MAP.keys())].mean(axis=1, skipna=True))
    for c in NCEMS_CRITERIA_ORDER:
        df[c] = np.nan
    for src_col, targets in HUMAN_NCEMS_COL_MAP.items():
        for t in targets:
            df[t] = df[src_col]
    df['review_text'] = df['rephrased_review'].fillna('').astype(str).str.strip()
    out = df[['id', 'title', 'title_norm', 'author', 'evaluator', 'overall_score', 'review_text', *NCEMS_CRITERIA_ORDER]].rename(columns={'id': 'proposal_id'})
    out['proposal_id'] = out['proposal_id'].astype(str)
    out['proposal_uid'] = out['author'].astype(str) + '::' + out['proposal_id'].astype(str)
    out.insert(0, 'source_file', path.name)
    out.insert(0, 'review_source', 'human')
    out.insert(0, 'review_type', 'ncems_criteria')
    out.insert(0, 'review_uid', [f'ncems-human-{cohort}-{i:04d}' for i in range(len(out))])
    return out


def build_novelty_ai_reviews():
    path = first_existing(NOVELTY_AI_REVIEWS_DIR.glob('novelty_reviews_*.json'), f'novelty AI reviews in {NOVELTY_AI_REVIEWS_DIR}')
    with open(path) as f:
        payload = json.load(f)
    rows = []
    for i, r in enumerate(payload.get('reviews', [])):
        ev = r.get('evaluations') or {}
        if not isinstance(ev, dict):
            continue
        criteria = {}
        justifications = []
        for c in NOVELTY_CRITERIA_ORDER:
            dim = ev.get(c)
            criteria[c] = dim.get('score', np.nan) if isinstance(dim, dict) else np.nan
            if isinstance(dim, dict) and isinstance(dim.get('justification'), str):
                justifications.append(dim['justification'].strip())
        overall_summary = ev.get('overall_summary', '') or ''
        review_text = overall_summary or '\n\n'.join([j for j in justifications if j])
        proposal_id = r.get('proposal_id') or ev.get('proposal_id')
        title = r.get('title')
        rows.append({
            'review_uid': f'novelty-ai-{i:04d}',
            'review_type': 'novelty',
            'review_source': 'ai',
            'source_file': path.name,
            'proposal_id': proposal_id,
            'title': title,
            'title_norm': normalize_title(title),
            'author': r.get('author'),
            'evaluator': r.get('evaluator'),
            'overall_score': ev.get('overall_novelty_score', np.nan),
            'review_text': review_text,
            **criteria,
        })
    df = pd.DataFrame(rows)
    if len(df):
        missing = df['proposal_id'].isna() | df['proposal_id'].astype(str).str.strip().str.lower().isin(['', 'n/a', 'na', 'none', 'nan'])
        df.loc[missing, 'proposal_id'] = df.loc[missing, 'title_norm']
        df['proposal_uid'] = df['author'].astype(str) + '::' + df['proposal_id'].astype(str)
    return df

ncems_reviews_df = pd.concat([build_ncems_ai_reviews(), build_ncems_human_reviews('y1'), build_ncems_human_reviews('y2')], ignore_index=True)
for c in ['overall_score', *NCEMS_CRITERIA_ORDER]:
    ncems_reviews_df[c] = pd.to_numeric(ncems_reviews_df[c], errors='coerce')
ncems_reviews_df.to_csv(NCEMS_ALL_REVIEWS_PATH, index=False)

novelty_reviews_df = build_novelty_ai_reviews()
for c in ['overall_score', *NOVELTY_CRITERIA_ORDER]:
    novelty_reviews_df[c] = pd.to_numeric(novelty_reviews_df[c], errors='coerce')
novelty_reviews_df.to_csv(NOVELTY_ALL_REVIEWS_PATH, index=False)

print(f'Saved NCEMS reviews: {NCEMS_ALL_REVIEWS_PATH} ({len(ncems_reviews_df)} rows)')
print(f'Saved novelty reviews: {NOVELTY_ALL_REVIEWS_PATH} ({len(novelty_reviews_df)} rows)')
print(ncems_reviews_df.groupby(['review_source', 'author']).size())
print(novelty_reviews_df.groupby(['review_source', 'author']).size())

Saved NCEMS reviews: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/prepared/ncems_criteria_all_reviews.csv (361 rows)
Saved novelty reviews: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/prepared/novelty_all_reviews.csv (276 rows)
review_source  author              
ai             claude-opus-4-5         69
               gemini-3-pro-preview    69
               gpt-5.2                 69
               human-y1                36
               human-y2                33
human          human-y1                47
               human-y2                38
dtype: int64
review_source  author              
ai             claude-opus-4-5         69
               gemini-3-pro-preview    69
               gpt-5.2                 69
               human-y1                36
               human-y2                33
dtype: int64


## 5. Save Review Scores Wide Table

In [6]:
NCEMS_FIELD_MAP = {
    'Relevance_to_Emergent_Phenomena': 'relevance_to_emergent_phenomena',
    'Novelty_and_Significance': 'novelty_and_significance',
    'Rigor_of_Approach': 'rigor_of_approach',
    'Scope_and_Timeline': 'scope_and_timeline',
    'Synthesis_Focus': 'synthesis_focus',
    'Data_Identification': 'data_identification',
    'Open_Science_Commitment': 'open_science_commitment',
}

ncems_ai = ncems_reviews_df[ncems_reviews_df['review_source'] == 'ai'].copy()
nov_ai = novelty_reviews_df[novelty_reviews_df['review_source'] == 'ai'].copy()

ncems_wide = (
    ncems_ai.groupby('title', as_index=False)[['overall_score', *NCEMS_CRITERIA_ORDER]]
    .mean()
    .rename(columns={'overall_score': 'review_score_mean', **NCEMS_FIELD_MAP})
)
nov_wide = (
    nov_ai.groupby('title', as_index=False)[['overall_score', *NOVELTY_CRITERIA_ORDER]]
    .mean()
    .rename(columns={'overall_score': 'novelty_score_mean'})
)

review_scores_wide = ncems_wide.merge(nov_wide, on='title', how='outer')
review_scores_wide.to_csv(REVIEW_SCORES_WIDE_PATH, index=False)
print(f'Saved review score means: {REVIEW_SCORES_WIDE_PATH} ({len(review_scores_wide)} proposals)')

Saved review score means: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/review_scores_wide.csv (92 proposals)


## 6. Embedding Helpers

In [7]:
MODEL_NAME = 'michiyasunaga/BioLinkBERT-large'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = None
embedding_model = None


def ensure_embedding_model_loaded():
    global tokenizer, embedding_model
    if tokenizer is not None and embedding_model is not None:
        return tokenizer, embedding_model
    print(f'Loading embedding model: {MODEL_NAME}')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    embedding_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    embedding_model.eval()
    print(f'Model loaded on: {device}')
    return tokenizer, embedding_model


def embed_texts(texts, batch_size=8, max_len=512, pooling='cls'):
    ensure_embedding_model_loaded()
    texts = ['' if pd.isna(t) else str(t) for t in texts]
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc='Embedding texts'):
            batch = texts[i:i + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors='pt',
            ).to(device)
            outputs = embedding_model(**encoded)
            if pooling == 'mean':
                attn = encoded['attention_mask'].unsqueeze(-1)
                masked = outputs.last_hidden_state * attn
                denom = attn.sum(dim=1).clamp(min=1)
                emb = masked.sum(dim=1) / denom
                emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            else:
                emb = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(emb.cpu().numpy())
    return np.vstack(all_embeddings)


def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def save_pickle(payload, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(payload, f)
    print(f'Saved: {path}')

## 7. Proposal Embeddings

In [8]:
ai_prop = proposal_df[proposal_df['is_ai']].reset_index(drop=True)
human_prop = proposal_df[~proposal_df['is_ai']].reset_index(drop=True)

ai_metadata = ai_prop[['model', 'title', 'group']].to_dict('records')
human_metadata = human_prop.rename(columns={'title': 'proposal_title'})[['proposal_title', 'group', 'source_file', 'cohort']].to_dict('records')


def embedding_cache_matches(payload, ai_texts, human_texts, text_key):
    return (
        payload.get('model_name') == MODEL_NAME
        and payload.get('text_field') == text_key
        and payload.get('ai_texts') == ai_texts
        and payload.get('human_texts') == human_texts
    )


def build_or_load_proposal_embeddings(path, ai_texts, human_texts, text_key, description):
    if path.exists():
        payload = load_pickle(path)
        if 'ai_embeddings' in payload and 'human_embeddings' in payload:
            print(f'Loaded existing {description}: {path}')
            if not embedding_cache_matches(payload, ai_texts, human_texts, text_key):
                print('  Note: cache metadata/text fingerprint differs; reusing existing embeddings to avoid recomputation.')
            return payload
        print(f'Existing {description} file is missing expected embedding keys; recomputing: {path}')
    print(f'Computing {description}...')
    payload = {
        'ai_embeddings': embed_texts(ai_texts, pooling='cls'),
        'human_embeddings': embed_texts(human_texts, pooling='cls'),
        'ai_metadata': ai_metadata,
        'human_metadata': human_metadata,
        'model_name': MODEL_NAME,
        'text_field': text_key,
        'ai_texts': ai_texts,
        'human_texts': human_texts,
        'timestamp': datetime.now().isoformat(),
    }
    save_pickle(payload, path)
    return payload

full_payload = build_or_load_proposal_embeddings(
    PROPOSAL_FULL_EMBEDDINGS_FILE,
    ai_prop['full_text'].fillna('').tolist(),
    human_prop['full_text'].fillna('').tolist(),
    'rephrased_full_text',
    'full rephrased proposal embeddings',
)
abstract_payload = build_or_load_proposal_embeddings(
    PROPOSAL_ABSTRACT_EMBEDDINGS_FILE,
    ai_prop['abstract_text'].fillna('').tolist(),
    human_prop['abstract_text'].fillna('').tolist(),
    'rephrased_abstract_only',
    'abstract-only rephrased proposal embeddings',
)

print('Full proposal embedding shapes:', full_payload['human_embeddings'].shape, full_payload['ai_embeddings'].shape)
print('Abstract proposal embedding shapes:', abstract_payload['human_embeddings'].shape, abstract_payload['ai_embeddings'].shape)

Loaded existing full rephrased proposal embeddings: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/embeddings/rephrased/minimal/proposal_embeddings_human_ai_rephrased.pkl
Loaded existing abstract-only rephrased proposal embeddings: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/embeddings/rephrased/minimal/proposal_embeddings_section1_only.pkl
Full proposal embedding shapes: (23, 1024) (69, 1024)
Abstract proposal embedding shapes: (23, 1024) (69, 1024)


## 8. Literature Embeddings

In [9]:
with open(LITERATURE_PATH) as f:
    literature_payload = json.load(f)
articles = literature_payload.get('articles', [])
corpus_texts = [f"Title: {a.get('title', '')}\n\nAbstract: {a.get('abstract', '')}" for a in articles]
corpus_metadata = [{
    'pmid': a.get('pmid'),
    'title': a.get('title'),
    'publication_date': a.get('publication_date'),
    'mesh_terms': a.get('mesh_terms', []),
} for a in articles]

if LITERATURE_EMBEDDINGS_FILE.exists():
    lit_payload = load_pickle(LITERATURE_EMBEDDINGS_FILE)
    if 'embeddings' in lit_payload:
        print(f'Loaded existing literature embeddings: {LITERATURE_EMBEDDINGS_FILE}')
        # Refresh lightweight metadata without recomputing embeddings.
        lit_payload['metadata'] = corpus_metadata
        lit_payload.setdefault('texts', corpus_texts)
        if lit_payload.get('model_name') != MODEL_NAME:
            print('  Note: cached model_name differs from current MODEL_NAME; reusing existing embeddings to avoid recomputation.')
        save_pickle(lit_payload, LITERATURE_EMBEDDINGS_FILE)
    else:
        print(f'Existing literature embedding file is missing expected embedding keys; recomputing: {LITERATURE_EMBEDDINGS_FILE}')
        lit_payload = None
else:
    lit_payload = None

if lit_payload is None:
    lit_payload = {
        'embeddings': embed_texts(corpus_texts, pooling='cls'),
        'metadata': corpus_metadata,
        'texts': corpus_texts,
        'model_name': MODEL_NAME,
        'timestamp': datetime.now().isoformat(),
    }
    save_pickle(lit_payload, LITERATURE_EMBEDDINGS_FILE)

print('Literature embedding shape:', np.asarray(lit_payload['embeddings']).shape)

Loading embedding model: michiyasunaga/BioLinkBERT-large
Model loaded on: cpu


Embedding texts: 100%|██████████| 4943/4943 [1:51:03<00:00,  1.35s/it]


Saved: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/embeddings/literature/relevant_literature_embeddings.pkl
Literature embedding shape: (39538, 1024)


## 9. Review Embeddings

In [10]:
def build_or_load_review_embeddings(df, path, description):
    texts = df['review_text'].fillna('').astype(str).tolist()
    review_uids = df['review_uid'].astype(str).tolist()
    metadata_cols = [c for c in ['review_uid', 'review_type', 'review_source', 'source_file', 'proposal_id', 'proposal_uid', 'title', 'title_norm', 'author', 'evaluator'] if c in df.columns]
    metadata = df[metadata_cols].to_dict('records')

    if path.exists():
        payload = load_pickle(path)
        if 'embeddings' in payload:
            print(f'Loaded existing {description}: {path}')
            if not (
                payload.get('model_name') == MODEL_NAME
                and payload.get('review_uids') == review_uids
                and payload.get('review_texts') == texts
            ):
                print('  Note: cache metadata/text fingerprint differs; reusing existing embeddings to avoid recomputation.')
            return payload
        print(f'Existing {description} file is missing expected embedding keys; recomputing: {path}')

    payload = {
        'embeddings': embed_texts(texts, pooling='mean'),
        'metadata': metadata,
        'review_uids': review_uids,
        'review_texts': texts,
        'model_name': MODEL_NAME,
        'pooling': 'attention_masked_mean_normalized',
        'timestamp': datetime.now().isoformat(),
    }
    save_pickle(payload, path)
    return payload

ncems_review_payload = build_or_load_review_embeddings(ncems_reviews_df, NCEMS_REVIEW_EMBEDDINGS_FILE, 'NCEMS review embeddings')
novelty_review_payload = build_or_load_review_embeddings(novelty_reviews_df, NOVELTY_REVIEW_EMBEDDINGS_FILE, 'novelty review embeddings')

print('NCEMS review embedding shape:', np.asarray(ncems_review_payload['embeddings']).shape)
print('Novelty review embedding shape:', np.asarray(novelty_review_payload['embeddings']).shape)

Existing NCEMS review embeddings file is missing expected embedding keys; recomputing: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/embeddings/reviews/minimal/ncems_criteria/review_embeddings_minimal.pkl


Embedding texts: 100%|██████████| 46/46 [00:18<00:00,  2.52it/s]


Saved: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/embeddings/reviews/minimal/ncems_criteria/review_embeddings_minimal.pkl


Embedding texts: 100%|██████████| 35/35 [00:21<00:00,  1.62it/s]

Saved: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/embeddings/reviews/minimal/novelty/review_embeddings_minimal.pkl
NCEMS review embedding shape: (361, 1024)
Novelty review embedding shape: (276, 1024)


## 10. Artifact Summary

In [11]:
artifact_paths = [
    PREPARED_ALL_PROPOSALS_PATH,
    PREPARED_ALL_PROPOSALS_CSV,
    ALL_PROPOSALS_PATH,
    NCEMS_ALL_REVIEWS_PATH,
    NOVELTY_ALL_REVIEWS_PATH,
    REVIEW_SCORES_WIDE_PATH,
    PROPOSAL_FULL_EMBEDDINGS_FILE,
    PROPOSAL_ABSTRACT_EMBEDDINGS_FILE,
    LITERATURE_EMBEDDINGS_FILE,
    NCEMS_REVIEW_EMBEDDINGS_FILE,
    NOVELTY_REVIEW_EMBEDDINGS_FILE,
]

print('Prepared artifacts:')
for p in artifact_paths:
    exists = p.exists()
    size = p.stat().st_size / 1024 / 1024 if exists else 0
    print(f'  {"OK" if exists else "MISSING"} {p} ({size:.2f} MB)')

Prepared artifacts:
  OK /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/prepared/all_proposals.json (2.23 MB)
  OK /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/prepared/all_proposals.csv (0.61 MB)
  OK /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/all_proposals.json (2.23 MB)
  OK /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/prepared/ncems_criteria_all_reviews.csv (0.33 MB)
  OK /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/prepared/novelty_all_reviews.csv (0.35 MB)
  OK /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/rephrased/minimal/review_scores_wide.csv (0.03 MB)
  OK /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/embeddings/rephrased/minimal/proposal_embeddings_human_ai_rephrased.pkl (0.53 MB)
  OK /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/embeddings/rephrased